#  `AIHUB 186.복지_분야_콜센터_상담데이터` 음성데이터 정보 획득

* 입력 폴더 위치: /home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터
* 출력 폴더 위치: /home/data/expr/week2/data_prepare_aihub
* 출력 파일: wav_info.tsv
  * 탭으로 구분된 csv 파일 (tsv)
  * 열
      * file_id
      * relative_path
      * duration_sec
      * format
      * sample_rate
      * channels
      * sample_width_bits
      * frame_count
      * file_size_bytes

### 입력폴더, 출력파일, 에러파일 정의

In [ ]:
input_dir = "/home/data/data/aihub/186.복지_분야_콜센터_상담데이터/01.데이터"
output_file = "/home/data/expr/week2/03-data_prepare_aihub/wav_info.tsv"
error_file = "/home/data/expr/week2/03-data_prepare_aihub/wav_errors.tsv"

### 라이브러리 로드

In [ ]:
import csv
import wave
from pathlib import Path

### 함수 정의

In [ ]:
def extract_wav_info_from_folder(
    input_dir: str,
    output_file: str = "wav_info.tsv",
    error_file: str = "wav_errors.tsv",
    include_relative_path: bool = False,
) -> None:
    """
    input_dir 이하의 모든 WAV 파일을 재귀적으로 탐색하여
    WAV 파일 정보를 TSV로 저장합니다.

    저장 정보:
    - file_id
    - relative_path
    - duration_sec
    - format
    - sample_rate
    - channels
    - sample_width_bits
    - frame_count
    - file_size_bytes

    표준 PCM WAV 파일을 대상으로 합니다.
    손상되었거나 wave 모듈에서 읽지 못하는 WAV 파일은
    오류 TSV에 별도로 기록합니다.
    """

    input_path = Path(input_dir).expanduser().resolve()
    output_path = Path(output_file).expanduser().resolve()
    error_path = Path(error_file).expanduser().resolve()

    if not input_path.exists():
        raise FileNotFoundError(f"입력 폴더가 없습니다: {input_path}")

    if not input_path.is_dir():
        raise NotADirectoryError(f"입력 경로가 폴더가 아닙니다: {input_path}")

    # 대소문자 확장자를 모두 처리
    wav_files = sorted(
        path
        for path in input_path.rglob("*")
        if path.is_file() and path.suffix.lower() == ".wav"
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    error_path.parent.mkdir(parents=True, exist_ok=True)

    success_count = 0
    error_count = 0
    total_duration_sec = 0.0

    with (
        output_path.open("w", encoding="utf-8-sig", newline="") as out_f,
        error_path.open("w", encoding="utf-8-sig", newline="") as err_f,
    ):
        writer = csv.writer(out_f, delimiter="\t")
        writer.writerow([
            "file_id",
            "relative_path",
            "duration_sec",
            "format",
            "sample_rate",
            "channels",
            "sample_width_bits",
            "frame_count",
            "file_size_bytes",
        ])

        error_writer = csv.writer(err_f, delimiter="\t")
        error_writer.writerow([
            "wav_path",
            "error_type",
            "error_message",
        ])

        for wav_path in wav_files:
            try:
                relative_path = str(wav_path.relative_to(input_path))

                if include_relative_path:
                    # 확장자를 포함한 상대경로를 파일 ID로 사용
                    file_id = relative_path
                else:
                    # 확장자를 제외한 파일명
                    file_id = wav_path.stem

                with wave.open(str(wav_path), "rb") as wav_f:
                    channels = wav_f.getnchannels()
                    sample_width_bytes = wav_f.getsampwidth()
                    sample_width_bits = sample_width_bytes * 8
                    sample_rate = wav_f.getframerate()
                    frame_count = wav_f.getnframes()
                    compression_type = wav_f.getcomptype()
                    compression_name = wav_f.getcompname()

                if sample_rate <= 0:
                    raise ValueError(
                        f"잘못된 샘플링레이트입니다: {sample_rate}"
                    )

                duration_sec = frame_count / sample_rate
                file_size_bytes = wav_path.stat().st_size

                if compression_type == "NONE":
                    wav_format = (
                        f"WAV PCM {sample_width_bits}-bit"
                    )
                else:
                    wav_format = (
                        f"WAV {compression_type} "
                        f"({compression_name})"
                    )

                writer.writerow([
                    file_id,
                    relative_path,
                    f"{duration_sec:.6f}",
                    wav_format,
                    sample_rate,
                    channels,
                    sample_width_bits,
                    frame_count,
                    file_size_bytes,
                ])

                success_count += 1
                total_duration_sec += duration_sec

            except (
                wave.Error,
                EOFError,
                OSError,
                ValueError,
            ) as e:
                error_count += 1

                print(f"[오류] {wav_path}: {e}")

                error_writer.writerow([
                    str(wav_path),
                    type(e).__name__,
                    str(e),
                ])

    total_duration_min = total_duration_sec / 60
    total_duration_hour = total_duration_sec / 3600

    print()
    print(f"입력 폴더: {input_path}")
    print(f"검색한 WAV 파일: {len(wav_files):,}개")
    print(f"정상 처리 파일: {success_count:,}개")
    print(f"오류 파일: {error_count:,}개")
    print(f"전체 길이: {total_duration_sec:,.3f}초")
    print(f"전체 길이: {total_duration_min:,.3f}분")
    print(f"전체 길이: {total_duration_hour:,.3f}시간")
    print(f"출력 파일: {output_path}")
    print(f"오류 파일: {error_path}")

### 음성 데이터 정보 추출 실행

In [ ]:
extract_wav_info_from_folder(
    input_dir=input_dir,
    output_file=output_file,
    error_file=error_file,
    include_relative_path=False,
)

### 음성 데이터 분포 확인

In [ ]:
text_file = "/home/data/expr/week2/03-data_prepare_aihub/orgtext.tsv"

In [ ]:
import pandas as pd

# TSV 읽기
text_df = pd.read_csv(
    text_file,
    sep="\t",
    encoding="utf-8-sig",
)

wav_df = pd.read_csv(
    output_file,
    sep="\t",
    encoding="utf-8-sig",
)

# file_id 기준 병합
df = text_df.merge(
    wav_df[["file_id", "duration_sec"]],
    left_on="file_label",
    right_on="file_id",
    how="inner",
)

# 같은 wav가 여러 문장으로 나뉘어 있을 수 있으므로
# file_label 기준으로 하나만 남김
df = df.drop_duplicates(subset="file_label")

### category1별 시간

In [ ]:
result = (
    df.groupby("category1")["duration_sec"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

result["sum_min"] = result["sum"] / 60
result["sum_hour"] = result["sum"] / 3600

print(result)

### category1 + category2별

In [ ]:
result = (
    df.groupby(["category1", "category2"])["duration_sec"]
      .agg(["count", "sum"])
      .reset_index()
)

result["sum_hour"] = result["sum"] / 3600
print(result)

### category1 + category2 + category3별

In [ ]:
result = (
    df.groupby(
        ["category1", "category2", "category3"]
    )["duration_sec"]
    .agg(["count", "sum"])
    .reset_index()
)

result["sum_hour"] = result["sum"] / 3600
print(result)